In [ ]:
%%sql -r dataframe_1
--Set Up a New Database Called INTL_DB
use role SYSADMIN;
create database INTL_DB;
use schema INTL_DB.PUBLIC;

In [ ]:
%%sql -r dataframe_2
--Create a Warehouse for Loading INTL_DB
use role SYSADMIN;

create warehouse INTL_WH 
with 
warehouse_size = 'XSMALL' 
warehouse_type = 'STANDARD' 
auto_suspend = 600 --600 seconds/10 mins
auto_resume = TRUE;

In [ ]:
%%sql -r dataframe_9
USE DATABASE intl_db;
USE SCHEMA public;

In [ ]:
%%sql -r dataframe_3
-- Create a Table and Load It
create or replace table intl_db.public.INT_STDS_ORG_3166 
(iso_country_name varchar(100), 
 country_name_official varchar(200), 
 sovreignty varchar(40), 
 alpha_code_2digit varchar(2), 
 alpha_code_3digit varchar(3), 
 numeric_country_code integer,
 iso_subdivision varchar(15), 
 internet_domain_code varchar(10)
);

In [ ]:
%%sql -r dataframe_4
create or replace file format util_db.public.PIPE_DBLQUOTE_HEADER_CR 
  type = 'CSV' --use CSV for any flat file
  compression = 'AUTO' 
  field_delimiter = '|' --pipe or vertical bar
  record_delimiter = '\r' --carriage return
  skip_header = 1  --1 header row
  field_optionally_enclosed_by = '\042'  --double quotes
  trim_space = FALSE;

In [ ]:
%%sql -r dataframe_5
show stages in account; 

In [ ]:
%%sql -r dataframe_7
copy into intl_db.public.INT_STDS_ORG_3166
from @util_db.public.MY_INTERNAL_STAGE
files = ( 'ISO_Countries_UTF8_pipe.csv')
file_format = ( format_name='util_db.public.PIPE_DBLQUOTE_HEADER_CR' );


In [ ]:
%%sql -r dataframe_8
select * from intl_db.public.INT_STDS_ORG_3166 
limit 10;

In [ ]:
%%sql -r dataframe_6
select count(*) as OBJECTS_FOUND
from INTL_DB.INFORMATION_SCHEMA.TABLES 
where table_schema = 'PUBLIC'
and table_name = 'INT_STDS_ORG_3166'

In [ ]:
%%sql -r dataframe_10
select row_count
from INTL_DB.INFORMATION_SCHEMA.TABLES 
where table_schema='PUBLIC'
and table_name= 'INT_STDS_ORG_3166';

In [ ]:
%%sql -r dataframe_11
select  
    iso_country_name
    ,country_name_official,alpha_code_2digit
    ,r_name as region
from intl_db.public.INT_STDS_ORG_3166 i
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION n
on upper(i.iso_country_name) = n.n_name
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION r
on n.n_regionkey = r.r_regionkey;

In [ ]:
%%sql -r dataframe_12
create view intl_db.public.NATIONS_SAMPLE_PLUS_ISO 
(  iso_country_name
    ,country_name_official
    ,alpha_code_2digit
    ,region
) AS
select  
    iso_country_name
    ,country_name_official,alpha_code_2digit
    ,r_name as region
from intl_db.public.INT_STDS_ORG_3166 i
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION n
on upper(i.iso_country_name) = n.n_name
left join SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION r
on n.n_regionkey = r.r_regionkey;

In [ ]:
%%sql -r dataframe_13
select *
from intl_db.public.NATIONS_SAMPLE_PLUS_ISO 
limit 10;

In [ ]:
%%sql -r dataframe_14
create table intl_db.public.CURRENCIES 
(
  currency_ID integer, 
  currency_char_code varchar(3), 
  currency_symbol varchar(4), 
  currency_digital_code varchar(3), 
  currency_digital_name varchar(30)
)
  comment = 'Information about currencies including character codes, symbols, digital codes, etc.';

In [ ]:
%%sql -r dataframe_15
create table intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE 
  (
    country_char_code varchar(3), 
    country_numeric_code integer, 
    country_name varchar(100), 
    currency_name varchar(100), 
    currency_char_code varchar(3), 
    currency_numeric_code integer
  ) 
  comment = 'Mapping table currencies to countries';

In [ ]:
%%sql -r dataframe_16
create or replace file format util_db.public.CSV_COMMA_LF_HEADER 
  type = 'CSV' --use CSV for any flat file
  field_delimiter = ',' 
  record_delimiter = '\n' --the n represents a Line Feed character
  skip_header = 1  --1 header row
;

In [ ]:
%%sql -r dataframe_17
copy into intl_db.public.CURRENCIES
from @util_db.public.MY_INTERNAL_STAGE
files = ( 'currencies.xls')
file_format = ( format_name='util_db.public.CSV_COMMA_LF_HEADER' );

In [ ]:
%%sql -r dataframe_18
copy into intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE
from @util_db.public.MY_INTERNAL_STAGE
files = ( 'country_code_to_currency_code.xls')
file_format = ( format_name='util_db.public.CSV_COMMA_LF_HEADER' );

In [ ]:
%%sql -r dataframe_19
create view simple_currency (
    CTY_CODE,
    CUR_CODE
) as
select country_char_code,
       currency_char_code
from intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE;

In [ ]:
%%sql -r dataframe_20
alter view INTL_DB.PUBLIC.NATIONS_SAMPLE_PLUS_ISO
set secure;

alter view INTL_DB.PUBLIC.SIMPLE_CURRENCY
set secure;